# Tencent WeMM-Embedding-9B + Qdrant — Kaggle T4×2 Production Demo

**VI:** Notebook public này giữ nguyên phần truy xuất semantic đã được chấp nhận và bổ sung một phần visual-retrieval riêng để chứng minh raw cosine `0.90+` một cách trung thực. Hai phần dùng **hai search space khác nhau** và được báo cáo riêng để tránh đánh đồng benchmark:

- **Semantic corpus retrieval:** Qdrant production corpus, `99,967` entities mỗi collection.
- **Visual robustness retrieval:** temporary curated gallery gồm `4` original images.

**EN:** This public notebook preserves the accepted semantic-retrieval showcase and adds a separate visual-retrieval section that demonstrates legitimate raw cosine `0.90+`. The two sections use **different search spaces** and are reported separately to avoid conflating benchmarks:

- **Semantic corpus retrieval:** production Qdrant corpus, `99,967` entities per collection.
- **Visual robustness retrieval:** temporary curated gallery containing `4` original images.

### Yêu cầu môi trường / Runtime requirements

- **Accelerator:** GPU T4 ×2
- **Internet:** ON
- **Dataset:** `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots`, version `1`
- **Model:** `dangkhoa2016/tencent-wemm-embedding-9b`, Transformers/default/version `1`

### Luồng trình diễn / Presentation flow

1. Steps 1–5 — system setup
2. Step 6 — bilingual text retrieval
3. Step 7A — semantic image→text cross-modal retrieval over the 99,967-entity corpus
4. Step 7B — transformed-image→original-image robustness retrieval over a 4-image temporary gallery
5. Step 8 — closeout, GPU reclaim, Qdrant storage seal


## Atomic Run All runner / Trình chạy Run All nguyên khối

**VI:** Toàn bộ workflow bootstrap → Steps 1–8 được thực thi trong **một code cell duy nhất**. Các tiêu đề và phần giải thích vẫn được render theo từng phase trong output, nhưng không còn code-cell boundary giữa các phase. Thiết kế này loại bỏ lỗi không ổn định của Kaggle khi một cell đã trả về `idle` nhưng Run All không submit cell code kế tiếp.

**EN:** The full bootstrap → Steps 1–8 workflow executes in **one code cell**. Section headings and explanations are still rendered phase-by-phase in the output, but there are no code-cell boundaries between phases. This removes the observed Kaggle failure mode where a cell returns `idle` successfully yet Run All does not submit the next code cell.


In [ ]:

from IPython.display import Markdown, display

print("ATOMIC_NOTEBOOK_RUNNER=START", flush=True)
print("ATOMIC_NOTEBOOK_CODE_BOUNDARIES=ELIMINATED", flush=True)

# Clean any previous notebook-owned demo session before touching runtime modules/files.
_existing_demo = globals().get("demo")
if _existing_demo is not None and not getattr(_existing_demo, "closed", True):
    print("ATOMIC_PREVIOUS_DEMO_CLEANUP=START", flush=True)
    _existing_demo.abort()
    print("ATOMIC_PREVIOUS_DEMO_CLEANUP=PASS", flush=True)
if "demo" in globals():
    del demo
del _existing_demo


display(Markdown("## Bootstrap runtime dùng lại / Bootstrap the reusable runtime\n\n**VI:** Checkout đúng Git commit, cài dependency trong chế độ output sạch, rồi chứng minh Python import đúng module từ checkout vừa tạo. Cảnh báo pip không liên quan sẽ không làm bẩn Saved Version; lỗi cài dependency thật sự vẫn fail-closed và in diagnostic đầy đủ.\n\n**EN:** Checkout the exact Git commit, install dependencies with clean output, and prove Python imports the runtime from that checkout. Unrelated pip resolver noise is suppressed; real dependency-install failures remain fail-closed with full diagnostics.\n"))

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REPO = 'https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU.git'
RUNTIME_COMMIT = 'd04bcd3e601b449b67d09ff1132cab965619d858'
ROOT = Path('/kaggle/working') / f'wemm-production-runtime-{RUNTIME_COMMIT[:12]}'

for name in list(sys.modules):
    if (
        name == 'wemm_runtime' or name.startswith('wemm_runtime.')
        or name == 'wemm_kaggle' or name.startswith('wemm_kaggle.')
    ):
        del sys.modules[name]

sys.path[:] = [
    entry for entry in sys.path
    if 'wemm-production-runtime' not in str(entry)
]
importlib.invalidate_caches()

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

subprocess.run(['git', 'init', '-q'], cwd=ROOT, check=True)
subprocess.run(['git', 'remote', 'add', 'origin', REPO], cwd=ROOT, check=True)
subprocess.run(
    ['git', 'fetch', '-q', '--depth', '1', 'origin', RUNTIME_COMMIT],
    cwd=ROOT,
    check=True,
)
subprocess.run(
    ['git', 'checkout', '-q', '--detach', 'FETCH_HEAD'],
    cwd=ROOT,
    check=True,
)

head = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'],
    cwd=ROOT,
    text=True,
).strip()
assert head == RUNTIME_COMMIT, (head, RUNTIME_COMMIT)

pip_result = subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        '--quiet',
        '--disable-pip-version-check',
        '-r', str(ROOT / 'requirements-kaggle.txt'),
        '-r', str(ROOT / 'requirements-demo.txt'),
    ],
    text=True,
    capture_output=True,
)
if pip_result.returncode != 0:
    if pip_result.stdout:
        print(pip_result.stdout, flush=True)
    if pip_result.stderr:
        print(pip_result.stderr, file=sys.stderr, flush=True)
    raise RuntimeError('Runtime dependency installation failed')

os.environ['WEMM_RUNTIME_SOURCE_COMMIT'] = RUNTIME_COMMIT
sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()

worker_module = importlib.import_module('wemm_runtime.worker')
worker_file = Path(worker_module.__file__).resolve()
assert ROOT.resolve() in worker_file.parents, (worker_file, ROOT)

print('Phụ thuộc runtime / Runtime dependencies : PASS')
print('Nguồn đã pin / Pinned source             : PASS')
print('Commit runtime / Runtime commit           : ' + RUNTIME_COMMIT)
print('Nguồn module thực thi / Import authority  : ' + str(worker_file))
print('PUBLIC_DEMO_SOURCE_BOOTSTRAP=PASS')
print('RUNTIME_SOURCE_COMMIT=' + RUNTIME_COMMIT)
print('RUNTIME_IMPORT_PATH=' + str(worker_file))

display(Markdown("## Steps 1/8–>5/8 — Chuẩn bị hệ thống / System setup\n\n**VI:** Cell này thực hiện hardware preflight, source verification, Dataset verification, Qdrant reuse/restore và load model worker. Nếu chạy lại trong cùng Kaggle kernel, notebook sẽ đóng sạch session `demo` cũ trước khi khởi tạo session mới để tránh stale worker/zombie PID. Sau khi cell kết thúc, Qdrant và GPU worker vẫn được giữ sống để các phần Step 6, Step 7A và Step 7B dùng lại mà không phải load/restore lại.\n\n**EN:** This cell performs hardware preflight, source verification, Dataset verification, Qdrant reuse/restore, and model-worker loading. When rerun in the same Kaggle kernel, the notebook cleanly closes any previous `demo` session before creating a new one, preventing stale worker/zombie-PID recovery failures. Qdrant and the GPU worker then remain alive for Steps 6, 7A, and 7B, avoiding redundant reload or restore work.\n"))

from wemm_kaggle.public_demo import start_public_demo

_previous_demo = globals().get("demo")
if _previous_demo is not None and not getattr(_previous_demo, "closed", True):
    print("PREVIOUS_DEMO_SESSION_CLEANUP=START", flush=True)
    _previous_demo.abort()
    print("PREVIOUS_DEMO_SESSION_CLEANUP=PASS", flush=True)
del _previous_demo

demo = start_public_demo()

display(Markdown("## Step 6/8 — Trình diễn truy vấn văn bản / Text query showcase\n\n**VI:** Phần này vẫn chạy nguyên 5 ví dụ EN↔VI đã khóa và cùng 20 retrieval paths như trước, nhưng chuyển sang cách trình bày **human-first**: mỗi thực thể hiển thị hai câu query, bảng kết quả 2×2 cho hai chiều EN→VI / VI→EN ở 4096d và 1024d, cùng khoảng cách tới đối thủ gần nhất. Top-3 raw cosine vẫn được giữ trong phần **Technical details / Chi tiết kỹ thuật** có thể mở rộng, và raw audit log đầy đủ vẫn được lưu riêng.\n\n**EN:** This section runs the same five frozen EN↔VI examples and the same 20 retrieval paths, but presents them **human-first**: each entity shows the two natural-language queries, a compact 2×2 result table for EN→VI / VI→EN at 4096d and 1024d, plus nearest-competitor separation. Raw-cosine Top-3 evidence remains available under expandable **Technical details**, and the complete raw audit log is still saved separately.\n"))

import contextlib
import html
import io
from pathlib import Path

from IPython.display import HTML, display


# Run the frozen Step 6 logic unchanged, but capture its verbose audit stdout.
_step6_stdout = io.StringIO()
with contextlib.redirect_stdout(_step6_stdout):
    text_results = demo.run_text()

_step6_raw_log = _step6_stdout.getvalue()
_step6_raw_path = Path("/kaggle/working/wemm-step6-bilingual-raw.log")
_step6_raw_path.write_text(_step6_raw_log, encoding="utf-8")


def _esc(value):
    return html.escape(str(value if value is not None else ""))


def _clip(value, limit=180):
    text = " ".join(str(value or "").split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def _nearest_competitor(path, expected_qid):
    for hit in path.get("top3", []):
        if str(hit.get("qid")) != str(expected_qid):
            return hit
    return None


def _path_map(item):
    mapped = {}
    for path in item["runtime_paths"]:
        direction = "EN→VI" if path["vector_name"] == "vi" else "VI→EN"
        mapped[(direction, int(path["dimension"]))] = path
    return mapped


def _result_cell(path, expected_qid):
    score = path.get("expected_score")
    rank = path.get("rank")
    competitor = _nearest_competitor(path, expected_qid)
    score_text = "n/a" if score is None else f"{float(score):.4f}"
    rank_text = "n/a" if rank is None else f"#{int(rank)}"
    if competitor is None or score is None:
        margin_html = "nearest competitor: n/a"
    else:
        margin = float(score) - float(competitor["score"])
        competitor_name = competitor.get("label_en") or competitor.get("label_vi") or competitor.get("qid")
        margin_html = (
            f"nearest: {_esc(_clip(competitor_name, 42))} "
            f"({_esc(competitor['qid'])}) · margin <b>+{margin:.4f}</b>"
        )
    status = "✅" if rank == 1 else "⚠️"
    return (
        f"<div style='font-size:15px'><b>{status} {rank_text} {_esc(expected_qid)}</b> · "
        f"raw cosine <b>{score_text}</b></div>"
        f"<div style='font-size:12px;opacity:.78;margin-top:3px'>{margin_html}</div>"
    )


def _technical_rows(item, paths):
    rows = []
    for direction in ("EN→VI", "VI→EN"):
        for dimension in (4096, 1024):
            path = paths[(direction, dimension)]
            for hit in path.get("top3", []):
                candidate = hit.get("label_en") or hit.get("label_vi") or hit.get("qid")
                marker = "✅" if str(hit.get("qid")) == str(item["qid"]) else ""
                rows.append(
                    "<tr>"
                    f"<td>{_esc(direction)} · {dimension}d</td>"
                    f"<td>#{int(hit['rank'])}</td>"
                    f"<td>{marker} <b>{_esc(hit['qid'])}</b> — {_esc(_clip(candidate, 92))}</td>"
                    f"<td style='text-align:right;font-family:monospace'>{float(hit['score']):.6f}</td>"
                    "</tr>"
                )
    return "".join(rows)


display(
    HTML(
        "<div style='padding:14px 16px;border:1px solid #c9c9c9;border-radius:12px;margin:8px 0 18px 0'>"
        "<div style='font-size:22px;font-weight:700'>Bilingual semantic retrieval / Truy xuất ngữ nghĩa song ngữ</div>"
        "<div style='margin-top:6px;line-height:1.55'>"
        "Each example asks one simple question: <b>can an English description retrieve the matching Vietnamese representation, "
        "and can the Vietnamese description independently retrieve the matching English representation?</b> "
        "The same test is repeated at 4096d and 1024d."
        "</div></div>"
    )
)

_top1_counts = {
    ("EN→VI", 4096): 0,
    ("EN→VI", 1024): 0,
    ("VI→EN", 4096): 0,
    ("VI→EN", 1024): 0,
}

for index, item in enumerate(text_results, 1):
    paths = _path_map(item)
    for key, path in paths.items():
        if path.get("rank") == 1:
            _top1_counts[key] += 1

    en4096 = paths[("EN→VI", 4096)]
    en1024 = paths[("EN→VI", 1024)]
    vi4096 = paths[("VI→EN", 4096)]
    vi1024 = paths[("VI→EN", 1024)]

    technical_rows = _technical_rows(item, paths)
    card = f"""
    <div style="border:1px solid #c9c9c9;border-radius:14px;padding:16px 18px;margin:14px 0 22px 0">
      <div style="font-size:13px;font-weight:700;letter-spacing:.04em;opacity:.72">
        EXAMPLE {index}/5 · CROSS-LANGUAGE SEMANTIC RETRIEVAL
      </div>
      <div style="font-size:24px;font-weight:750;margin-top:3px">
        {_esc(item['name'])} <span style="font-size:15px;font-weight:500;opacity:.7">({_esc(item['qid'])})</span>
      </div>
      <div style="font-size:13px;opacity:.72;margin:2px 0 14px 0">
        {_esc(item['category'])}
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
        <div style="border:1px solid #dddddd;border-radius:10px;padding:12px">
          <div style="font-weight:700;margin-bottom:6px">🇬🇧 English query</div>
          <div style="line-height:1.48">{_esc(item['text_en'])}</div>
          <div style="margin-top:10px;font-size:12px;opacity:.72">
            English meaning → embedding → search <b>Vietnamese vectors</b>
          </div>
        </div>
        <div style="border:1px solid #dddddd;border-radius:10px;padding:12px">
          <div style="font-weight:700;margin-bottom:6px">🇻🇳 Vietnamese query</div>
          <div style="line-height:1.48">{_esc(item['text_vi'])}</div>
          <div style="margin-top:10px;font-size:12px;opacity:.72">
            Vietnamese meaning → embedding → search <b>English vectors</b>
          </div>
        </div>
      </div>

      <div style="font-size:16px;font-weight:700;margin:16px 0 7px 0">
        Cross-language result / Kết quả truy xuất chéo ngôn ngữ
      </div>
      <table style="width:100%;border-collapse:collapse;font-size:14px">
        <thead>
          <tr>
            <th style="text-align:left;padding:8px;border-bottom:1px solid #cccccc">Direction</th>
            <th style="text-align:left;padding:8px;border-bottom:1px solid #cccccc">4096d</th>
            <th style="text-align:left;padding:8px;border-bottom:1px solid #cccccc">1024d</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td style="padding:9px 8px;vertical-align:top"><b>🇬🇧 EN → 🇻🇳 VI</b></td>
            <td style="padding:9px 8px;vertical-align:top">{_result_cell(en4096, item['qid'])}</td>
            <td style="padding:9px 8px;vertical-align:top">{_result_cell(en1024, item['qid'])}</td>
          </tr>
          <tr>
            <td style="padding:9px 8px;vertical-align:top"><b>🇻🇳 VI → 🇬🇧 EN</b></td>
            <td style="padding:9px 8px;vertical-align:top">{_result_cell(vi4096, item['qid'])}</td>
            <td style="padding:9px 8px;vertical-align:top">{_result_cell(vi1024, item['qid'])}</td>
          </tr>
        </tbody>
      </table>

      <div style="margin-top:14px;padding:11px 12px;border:1px solid #dddddd;border-radius:10px;line-height:1.5">
        <b>What happened? / Điều gì vừa xảy ra?</b><br>
        The English description retrieved <b>{_esc(item['qid'])} — {_esc(item['name'])}</b> at rank #1
        from Vietnamese vectors, and the Vietnamese description independently retrieved the same entity at rank #1
        from English vectors. The result held at both <b>4096d</b> and <b>1024d</b>.
      </div>

      <div style="margin-top:12px;font-size:18px;font-weight:750">
        ✅ 4 / 4 cross-language paths TOP-1 — PASS
      </div>

      <details style="margin-top:14px">
        <summary style="cursor:pointer;font-weight:700">
          Technical details / Chi tiết kỹ thuật — Top-3 raw cosine
        </summary>
        <div style="margin-top:8px">
          <table style="width:100%;border-collapse:collapse;font-size:12px">
            <thead>
              <tr>
                <th style="text-align:left;padding:6px;border-bottom:1px solid #cccccc">Path</th>
                <th style="text-align:left;padding:6px;border-bottom:1px solid #cccccc">Rank</th>
                <th style="text-align:left;padding:6px;border-bottom:1px solid #cccccc">Candidate</th>
                <th style="text-align:right;padding:6px;border-bottom:1px solid #cccccc">Raw cosine</th>
              </tr>
            </thead>
            <tbody>{technical_rows}</tbody>
          </table>
          <div style="font-size:12px;opacity:.72;margin-top:7px">
            Raw cosine is shown as a similarity score, not as a confidence percentage.
          </div>
        </div>
      </details>
    </div>
    """
    display(HTML(card))

_total_top1 = sum(_top1_counts.values())
if len(text_results) != 5 or _total_top1 != 20:
    raise RuntimeError(
        f"Human-first Step 6 summary contract failed: "
        f"entities={len(text_results)} top1_paths={_total_top1}"
    )

summary_html = f"""
<div style="border:2px solid #b9b9b9;border-radius:14px;padding:16px 18px;margin:18px 0">
  <div style="font-size:23px;font-weight:750">Bilingual retrieval summary / Tổng kết truy xuất song ngữ</div>
  <div style="margin:7px 0 13px 0">
    <b>5 real entities</b> · <b>10 natural-language embeddings</b> · <b>20 cross-language retrieval paths</b>
  </div>
  <table style="width:100%;border-collapse:collapse;font-size:15px">
    <thead>
      <tr>
        <th style="text-align:left;padding:8px;border-bottom:1px solid #cccccc">Direction</th>
        <th style="text-align:center;padding:8px;border-bottom:1px solid #cccccc">4096d</th>
        <th style="text-align:center;padding:8px;border-bottom:1px solid #cccccc">1024d</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="padding:9px 8px"><b>🇬🇧 EN → 🇻🇳 VI</b></td>
        <td style="text-align:center;padding:9px 8px"><b>{_top1_counts[('EN→VI', 4096)]}/5 ✅</b></td>
        <td style="text-align:center;padding:9px 8px"><b>{_top1_counts[('EN→VI', 1024)]}/5 ✅</b></td>
      </tr>
      <tr>
        <td style="padding:9px 8px"><b>🇻🇳 VI → 🇬🇧 EN</b></td>
        <td style="text-align:center;padding:9px 8px"><b>{_top1_counts[('VI→EN', 4096)]}/5 ✅</b></td>
        <td style="text-align:center;padding:9px 8px"><b>{_top1_counts[('VI→EN', 1024)]}/5 ✅</b></td>
      </tr>
    </tbody>
  </table>
  <div style="font-size:21px;font-weight:800;margin-top:14px">✅ TOTAL: 20 / 20 TOP-1</div>
  <div style="margin-top:6px;line-height:1.5">
    Every English query retrieved the correct Vietnamese representation, and every Vietnamese query retrieved
    the correct English representation, at both embedding dimensions. No threshold relaxation was used.
  </div>
</div>
"""
display(HTML(summary_html))

# Re-emit the machine-readable acceptance lines produced by the unchanged runtime.
_marker_prefixes = (
    "FROZEN_BILINGUAL_SHOWCASE=",
    "TEXT_SHOWCASE_ENTITIES=",
    "TEXT_SHOWCASE_EMBEDDING_REQUESTS=",
    "TEXT_SHOWCASE_QUERY_EXECUTIONS=",
    "TEXT_SHOWCASE_LANGUAGE_GATE_AUTHORITY=",
    "TEXT_SHOWCASE_ALL_TOP1=",
    "TEXT_SHOWCASE_STRICT_ALL_TOP3=",
    "TEXT_SHOWCASE_THRESHOLD_RELAXED=",
)
for _line in _step6_raw_log.splitlines():
    if _line.startswith(_marker_prefixes):
        print(_line, flush=True)

print(f"STEP_6_RAW_AUDIT_LOG={_step6_raw_path}", flush=True)
print("STEP_6_PRESENTATION_LAYER=HUMAN_FIRST", flush=True)
print("STEP_6_NOTEBOOK_CELL_RETURN=PASS", flush=True)
print("RUN_ALL_CONTINUATION_READY_FOR_STEP_7A=PASS", flush=True)

display(Markdown("## Step 7A/8 — Truy xuất semantic đa phương thức / Semantic cross-modal retrieval\n\n**VI:** Đây là phần image→text gốc và vẫn được giữ nguyên. Nó chứng minh ảnh có thể truy xuất đúng entity text tiếng Anh và tiếng Việt ở cả 4096d và 1024d. Raw cosine của cross-modal space được hiển thị nguyên bản, không được rescale.\n\n**EN:** This is the original image→text showcase and remains unchanged. It proves that images retrieve the correct English and Vietnamese entity text at both 4096d and 1024d. Raw cross-modal cosine values are shown as-is and are never rescaled.\n"))

print("STEP_7A_NOTEBOOK_CELL_ENTER=PASS", flush=True)
image_results = demo.run_image()

display(Markdown("## Step 7B/8 — Truy xuất hình ảnh độ tin cậy cao / High-confidence visual retrieval\n\n**VI:** Phần này dùng **4 winner được curate từ run độc lập 24-candidate**. Tiêu chí chọn không phải “4 score cao nhất”, mà là:\n\n- strict rank `#1`;\n- raw cosine tối thiểu qua 8 path `>= 0.90`;\n- có nhãn VI/EN rõ;\n- ảnh và entity dễ hiểu bằng mắt;\n- đủ đa dạng để public demo không giống một benchmark nhân tạo.\n\nBốn ví dụ được chọn:\n\n| QID | VI | EN | Min cosine ở curation run |\n|---|---|---|---:|\n| `Q19217` | Lâm Trịnh Nguyệt Nga | Carrie Lam | `0.980786` |\n| `Q10489198` | Trường Đại học Sư phạm Thành phố Hồ Chí Minh | Ho Chi Minh University of Education | `0.969598` |\n| `Q168751` | Đại học Duke | Duke University | `0.963304` |\n| `Q51756` | China Airlines | China Airlines | `0.950137` |\n\nSearch space của Step 7B là **temporary gallery gồm đúng 4 original P18 images**; nó không phải corpus Qdrant 99,967 entities. Mỗi original P18 được embed một lần. Sau đó notebook tạo 4 query transformations thực sự:\n\n- resize `80%`;\n- JPEG quality `90`;\n- center crop `96%`;\n- brightness `103%`.\n\nMỗi query được tìm ở cả `4096d` và `1024d`, tức **32 retrieval paths**. PASS chỉ khi **toàn bộ 32/32 paths rank #1 và raw cosine >= 0.90**.\n\n**EN:** This section uses four presentation-quality winners selected from the independent 24-candidate curation run. It performs 4 transformations × 2 dimensions × 4 entities = **32 retrieval paths**. PASS requires all 32 paths to rank the correct original image at #1 with **raw cosine >= 0.90**. No score rescaling and no threshold relaxation are permitted.\n"))

import hashlib
import io
import json
import math
import shutil
import time
from pathlib import Path
from urllib.parse import quote

import requests
from PIL import Image, ImageEnhance, ImageOps
from IPython.display import display
from qdrant_client import QdrantClient, models

from wemm_kaggle.demo_config import EVID, RUN_ROOT
from wemm_kaggle.demo_io import truncate_vector
from wemm_kaggle.demo_qdrant import write_cache_seal

visual_phase_t0 = time.perf_counter()

if demo.closed:
    raise RuntimeError("Demo session is already closed")
if demo.image_results is None:
    raise RuntimeError("Step 7A must complete before Step 7B")

VISUAL_THRESHOLD = 0.90
VISUAL_MAX_EDGE = 768
VISUAL_MAX_BYTES = 20 * 1024 * 1024
VISUAL_ROOT = RUN_ROOT / "visual-robustness"
VISUAL_IMAGES = VISUAL_ROOT / "images"
VISUAL_TRANSFORMS = VISUAL_ROOT / "transforms"
VISUAL_QDRANT = VISUAL_ROOT / "qdrant-local"

if VISUAL_ROOT.exists():
    shutil.rmtree(VISUAL_ROOT)
VISUAL_IMAGES.mkdir(parents=True)
VISUAL_TRANSFORMS.mkdir(parents=True)

VISUAL_SPECS = [
    {
        "qid": "Q19217",
        "label_vi": "Lâm Trịnh Nguyệt Nga",
        "label_en": "Carrie Lam",
        "p18": "Carrie Lam 2019-04-09 (1).jpg",
        "curation_min_score": 0.980786,
    },
    {
        "qid": "Q10489198",
        "label_vi": "Trường Đại học Sư phạm Thành phố Hồ Chí Minh",
        "label_en": "Ho Chi Minh University of Education",
        "p18": "Ho Chi Minh City Pedagogical University August 30, 2018.jpg",
        "curation_min_score": 0.969598,
    },
    {
        "qid": "Q168751",
        "label_vi": "Đại học Duke",
        "label_en": "Duke University",
        "p18": "Duke Chapel 4 16 05.jpg",
        "curation_min_score": 0.963304,
    },
    {
        "qid": "Q51756",
        "label_vi": "China Airlines",
        "label_en": "China Airlines",
        "p18": "B-18902 A350-900 China Airlines LHR 4.11.20.jpg",
        "curation_min_score": 0.950137,
    },
]

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Tencent-WeMM-Embedding-9B/1.0 visual-robustness public demo "
        "(https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU)"
    )
})

def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _download_normalized(spec: dict) -> tuple[Path, dict]:
    url = (
        "https://commons.wikimedia.org/wiki/Special:Redirect/file/"
        + quote(spec["p18"], safe="")
        + f"?width={VISUAL_MAX_EDGE}"
    )
    response = session.get(url, timeout=90)
    response.raise_for_status()
    raw = response.content
    if len(raw) > VISUAL_MAX_BYTES:
        raise RuntimeError(f"{spec['qid']}: image exceeds byte bound")

    with Image.open(io.BytesIO(raw)) as source:
        source.load()
        source = ImageOps.exif_transpose(source)
        rgb0 = source.convert("RGB")
        fresh = Image.new("RGB", rgb0.size)
        fresh.paste(rgb0)

    source_size = list(map(int, fresh.size))
    fresh.thumbnail(
        (VISUAL_MAX_EDGE, VISUAL_MAX_EDGE),
        Image.Resampling.LANCZOS,
    )
    path = VISUAL_IMAGES / f"{spec['qid']}.png"
    fresh.save(path, format="PNG", compress_level=0)

    return path, {
        "download_url": url,
        "resolved_url": response.url,
        "downloaded_bytes": len(raw),
        "source_size": source_size,
        "normalized_size": list(map(int, fresh.size)),
        "normalized_png_sha256": _sha256(path),
    }

def _make_transforms(qid: str, source_path: Path) -> dict[str, Path]:
    out_dir = VISUAL_TRANSFORMS / qid
    out_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(source_path) as source:
        base = source.convert("RGB")

    width, height = base.size
    paths = {}

    p = out_dir / "resize_80pct.png"
    base.resize(
        (max(64, int(width * 0.8)), max(64, int(height * 0.8))),
        Image.Resampling.LANCZOS,
    ).save(p, "PNG", compress_level=0)
    paths["resize_80pct"] = p

    p = out_dir / "jpeg_q90.jpg"
    base.save(p, "JPEG", quality=90, optimize=False, subsampling=0)
    paths["jpeg_q90"] = p

    dx = max(1, int(width * 0.02))
    dy = max(1, int(height * 0.02))
    cropped = base.crop((dx, dy, width - dx, height - dy)).resize(
        (width, height),
        Image.Resampling.LANCZOS,
    )
    p = out_dir / "center_crop_96pct.png"
    cropped.save(p, "PNG", compress_level=0)
    paths["center_crop_96pct"] = p

    p = out_dir / "brightness_103pct.png"
    ImageEnhance.Brightness(base).enhance(1.03).save(
        p, "PNG", compress_level=0
    )
    paths["brightness_103pct"] = p
    return paths

local_client = None
visual_results = None
try:
    assets = []
    for spec in VISUAL_SPECS:
        path, asset = _download_normalized(spec)
        assets.append({**spec, "image_path": str(path), "asset": asset})

    local_client = QdrantClient(path=str(VISUAL_QDRANT))
    local_client.create_collection(
        collection_name="visual_originals",
        vectors_config={
            "image_4096": models.VectorParams(
                size=4096, distance=models.Distance.COSINE
            ),
            "image_1024": models.VectorParams(
                size=1024, distance=models.Distance.COSINE
            ),
        },
    )

    for rec in assets:
        vector_4096 = demo.worker.embed_image(rec["image_path"], 4096)
        vector_1024 = truncate_vector(vector_4096, 1024)
        local_client.upsert(
            collection_name="visual_originals",
            points=[
                models.PointStruct(
                    id=int(rec["qid"][1:]),
                    vector={
                        "image_4096": vector_4096,
                        "image_1024": vector_1024,
                    },
                    payload={
                        "qid": rec["qid"],
                        "label_vi": rec["label_vi"],
                        "label_en": rec["label_en"],
                        "p18": rec["p18"],
                        "original_sha256": rec["asset"]["normalized_png_sha256"],
                    },
                )
            ],
            wait=True,
        )

    assert int(local_client.count("visual_originals", exact=True).count) == 4

    rows = []
    total_paths = 0
    passed_paths = 0

    for index, rec in enumerate(assets, 1):
        qid = rec["qid"]
        transform_paths = _make_transforms(qid, Path(rec["image_path"]))
        path_results = []

        print("\n" + "-" * 92)
        print(
            f"[HÌNH ẢNH ĐỘ TIN CẬY CAO / HIGH-CONFIDENCE VISUAL {index}/4] "
            f"{rec['label_vi']} / {rec['label_en']} — {qid}",
            flush=True,
        )
        print(f"  P18: {rec['p18']}", flush=True)
        print(
            "  SHA-256 ảnh gốc / Original SHA-256: "
            + rec["asset"]["normalized_png_sha256"],
            flush=True,
        )
        print(
            "  Kích thước chuẩn hóa / Normalized size: "
            + str(rec["asset"]["normalized_size"]),
            flush=True,
        )
        print(
            "  Min cosine ở curation run / Curation reference min cosine: "
            f"{rec['curation_min_score']:.6f}",
            flush=True,
        )

        print("  Ảnh gốc / Original preview:", flush=True)
        with Image.open(rec["image_path"]) as shown:
            thumb = shown.copy()
            thumb.thumbnail((520, 360), Image.Resampling.LANCZOS)
            display(thumb)

        representative = transform_paths["center_crop_96pct"]
        print(
            "  Ảnh biến đổi / Transformed preview — center_crop_96pct:",
            flush=True,
        )
        with Image.open(representative) as shown:
            thumb = shown.copy()
            thumb.thumbnail((520, 360), Image.Resampling.LANCZOS)
            display(thumb)

        for transform_name, transform_path in transform_paths.items():
            query_4096 = demo.worker.embed_image(transform_path, 4096)
            query_1024 = truncate_vector(query_4096, 1024)

            transform_sha = _sha256(transform_path)
            for dimension, query_vector in (
                (4096, query_4096),
                (1024, query_1024),
            ):
                response = local_client.query_points(
                    collection_name="visual_originals",
                    query=query_vector,
                    using=f"image_{dimension}",
                    limit=4,
                    with_payload=True,
                )
                points = list(response.points)
                expected_rank = next(
                    (
                        rank
                        for rank, point in enumerate(points, 1)
                        if str((point.payload or {}).get("qid", "")) == qid
                    ),
                    None,
                )
                expected_score = next(
                    (
                        float(point.score)
                        for point in points
                        if str((point.payload or {}).get("qid", "")) == qid
                    ),
                    None,
                )
                path_pass = (
                    expected_rank == 1
                    and expected_score is not None
                    and expected_score >= VISUAL_THRESHOLD
                )
                total_paths += 1
                passed_paths += int(path_pass)

                top3 = []
                for rank, point in enumerate(points[:3], 1):
                    payload = dict(point.payload or {})
                    top3.append({
                        "rank": rank,
                        "qid": str(payload.get("qid", "")),
                        "score": float(point.score),
                        "label_vi": str(payload.get("label_vi", "")),
                        "label_en": str(payload.get("label_en", "")),
                    })

                path_results.append({
                    "transform": transform_name,
                    "dimension": dimension,
                    "transform_sha256": transform_sha,
                    "expected_rank": expected_rank,
                    "raw_cosine": expected_score,
                    "pass": path_pass,
                    "top3": top3,
                })

                print(
                    f"  {transform_name:<20} | {dimension:4d}d | "
                    f"rank=#{expected_rank} | raw cosine={expected_score:.6f} | "
                    f"{'PASS' if path_pass else 'FAIL'}",
                    flush=True,
                )
                for hit in top3:
                    print(
                        f"      #{hit['rank']} {hit['qid']:<11} "
                        f"score={hit['score']:.6f} | "
                        f"VI={hit['label_vi']} | EN={hit['label_en']}",
                        flush=True,
                    )

        scores = [p["raw_cosine"] for p in path_results]
        strict_pass = len(path_results) == 8 and all(p["pass"] for p in path_results)
        row = {
            "qid": qid,
            "label_vi": rec["label_vi"],
            "label_en": rec["label_en"],
            "p18": rec["p18"],
            "asset": rec["asset"],
            "curation_reference_min_cosine": rec["curation_min_score"],
            "runtime_paths": path_results,
            "runtime_min_raw_cosine": min(scores),
            "strict_pass": strict_pass,
        }
        rows.append(row)
        print(
            "  Min raw cosine runtime / Runtime minimum raw cosine: "
            f"{row['runtime_min_raw_cosine']:.6f}",
            flush=True,
        )
        print(
            "  Kết quả / Verdict: "
            + ("8/8 PATHS TOP-1 >= 0.90 — PASS" if strict_pass else "FAIL"),
            flush=True,
        )

    runtime_min = min(row["runtime_min_raw_cosine"] for row in rows)
    all_strict = (
        total_paths == 32
        and passed_paths == 32
        and all(row["strict_pass"] for row in rows)
    )

    visual_results = {
        "verdict": "PASS" if all_strict else "FAIL",
        "threshold": VISUAL_THRESHOLD,
        "raw_cosine_rescaled": False,
        "threshold_relaxed": False,
        "production_collections_mutated": False,
        "example_count": len(rows),
        "total_paths": total_paths,
        "passed_paths": passed_paths,
        "runtime_min_raw_cosine": runtime_min,
        "examples": rows,
    }
    (EVID / "visual-robustness.json").write_text(
        json.dumps(visual_results, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("\n" + "=" * 92)
    print("TỔNG KẾT TRUY XUẤT HÌNH ẢNH ĐỘ TIN CẬY CAO / HIGH-CONFIDENCE VISUAL SCORECARD")
    print("=" * 92)
    print(f"Ví dụ / Examples                         : {len(rows)}/4")
    print(f"Đường truy xuất TOP-1 / TOP-1 paths      : {passed_paths}/{total_paths}")
    print(f"Raw cosine thấp nhất / Minimum raw cosine : {runtime_min:.6f}")
    print(f"Ngưỡng strict / Strict threshold          : {VISUAL_THRESHOLD:.2f}")
    print("Rescale raw cosine                         : NO / KHÔNG")
    print("Nới ngưỡng / Threshold relaxation          : NO / KHÔNG")
    print("Thay đổi production collections            : NO / KHÔNG")

    if not all_strict:
        raise RuntimeError(
            f"Visual robustness failed: passed={passed_paths}/{total_paths}, "
            f"min_raw_cosine={runtime_min:.6f}"
        )

    print("HIGH_CONFIDENCE_VISUAL_RETRIEVAL=PASS")
    print("VISUAL_RETRIEVAL_EXAMPLES=4")
    print("VISUAL_RETRIEVAL_PATHS_TOP1=32/32")
    print(f"VISUAL_RETRIEVAL_MIN_RAW_COSINE={runtime_min:.6f}")
    print("VISUAL_RETRIEVAL_THRESHOLD=0.90")
    print("VISUAL_RETRIEVAL_RAW_COSINE_RESCALED=NO")
    print("VISUAL_RETRIEVAL_THRESHOLD_RELAXED=NO")
    print("VISUAL_RETRIEVAL_PRODUCTION_COLLECTIONS_MUTATED=NO")

except BaseException:
    try:
        if local_client is not None:
            local_client.close()
    finally:
        if VISUAL_QDRANT.exists():
            shutil.rmtree(VISUAL_QDRANT, ignore_errors=True)
    # Preserve production reuse semantics even when Step 7B fails.
    if not demo.closed:
        demo.abort()
        try:
            write_cache_seal(demo.data_mode)
        except Exception:
            pass
    raise
finally:
    if local_client is not None:
        try:
            local_client.close()
        except Exception:
            pass
    if VISUAL_QDRANT.exists():
        shutil.rmtree(VISUAL_QDRANT, ignore_errors=True)

demo.phase_times["7b_visual_robustness"] = time.perf_counter() - visual_phase_t0
print(
    "Thời gian Step 7B / Step 7B time              : "
    f"{demo.phase_times['7b_visual_robustness']:.3f} s"
)
print("VISUAL_RETRIEVAL_TEMP_QDRANT=DELETED")

display(Markdown("## Step 8/8 — Đóng phiên + nghiệm thu / Closeout + acceptance\n\n**VI:** Cell cuối giữ nguyên closeout authority hiện có: giải phóng GPU worker, xác minh VRAM reclaim, dừng và seal Qdrant production storage. Scorecard cuối báo cáo **hai search space độc lập**:\n\n- semantic corpus retrieval: `36/36 TOP-1` trên corpus `99,967` entities;\n- visual robustness retrieval: `32/32 TOP-1` trên temporary `4-image gallery`.\n\nCon số `68/68` chỉ được dùng với tên **TOTAL EXECUTED RETRIEVAL CHECKS**, không được diễn đạt như thể cả 68 path đều chạy trên cùng corpus production.\n\n**EN:** The final cell preserves the existing closeout authority: release the GPU worker, verify VRAM reclaim, stop and seal production Qdrant storage. The final scorecard reports **two independent search spaces**:\n\n- semantic corpus retrieval: `36/36 TOP-1` over the `99,967`-entity corpus;\n- visual robustness retrieval: `32/32 TOP-1` over a temporary `4-image gallery`.\n\nThe `68/68` aggregate is labeled **TOTAL EXECUTED RETRIEVAL CHECKS** only; it is not presented as if all 68 paths searched the same production corpus.\n"))

if visual_results is None or visual_results.get("verdict") != "PASS":
    raise RuntimeError("Step 7B must PASS before Step 8 closeout")

final_summary = demo.closeout()

assert visual_results["total_paths"] == 32
assert visual_results["passed_paths"] == 32

print("\n" + "=" * 92)
print("BẢNG TỔNG KẾT MỞ RỘNG / EXTENDED PUBLIC DEMO SCORECARD")
print("=" * 92)

print("\n[1] TRUY XUẤT CORPUS SEMANTIC / SEMANTIC CORPUS RETRIEVAL")
print("Search space                              : 99,967 entities / collection")
print("Bilingual text + semantic image→text      : 36/36 TOP-1")

print("\n[2] ĐỘ BỀN TRUY XUẤT HÌNH ẢNH / VISUAL ROBUSTNESS RETRIEVAL")
print("Search space                              : 4-image temporary curated gallery")
print("Transformed image→original image          : 32/32 TOP-1")
print(
    "Minimum visual raw cosine                : "
    f"{visual_results['runtime_min_raw_cosine']:.6f}"
)
print("Visual raw cosine threshold               : 0.90")
print("Raw cosine rescaled                       : NO")
print("Threshold relaxed                         : NO")

print("\n[3] TỔNG KIỂM TRA ĐÃ THỰC THI / TOTAL EXECUTED RETRIEVAL CHECKS")
print("Executed retrieval checks                 : 68/68 PASS")
print("NOTE: the 36 semantic and 32 visual checks use different search spaces.")

print("PUBLIC_DEMO_EXTENDED_SCORECARD=PASS")
print("SEMANTIC_CORPUS_RETRIEVAL_PATHS_TOP1=36/36")
print("SEMANTIC_CORPUS_SEARCH_SPACE_ENTITIES=99967")
print("VISUAL_ROBUSTNESS_RETRIEVAL_PATHS_TOP1=32/32")
print("VISUAL_ROBUSTNESS_SEARCH_SPACE_IMAGES=4")
print("PUBLIC_DEMO_TOTAL_EXECUTED_RETRIEVAL_CHECKS=68/68")

print("ATOMIC_NOTEBOOK_RUNNER=PASS", flush=True)
display(Markdown("## Contract chấp nhận cuối cùng / Final acceptance contract\n\n**VI:** `Run All` thành công phải đi tuần tự qua setup → text → Step 7A semantic image→text → Step 7B visual robustness → closeout. Hai benchmark được giữ tách biệt về search space.\n\n**EN:** A successful `Run All` must proceed through setup → text → Step 7A semantic image→text → Step 7B visual robustness → closeout. The two benchmarks remain explicitly separated by search space.\n\n```text\nFROZEN_BILINGUAL_SHOWCASE=PASS\nTEXT_SHOWCASE_ALL_TOP1=5/5\nTEXT_SHOWCASE_STRICT_ALL_TOP3=PASS\n\nFROZEN_SHOWCASE=PASS\nFROZEN_SHOWCASE_COUNT=4\nFROZEN_SHOWCASE_STRICT_ALL_TOP3=PASS\n\nHIGH_CONFIDENCE_VISUAL_RETRIEVAL=PASS\nVISUAL_RETRIEVAL_EXAMPLES=4\nVISUAL_RETRIEVAL_PATHS_TOP1=32/32\nVISUAL_RETRIEVAL_THRESHOLD=0.90\nVISUAL_RETRIEVAL_RAW_COSINE_RESCALED=NO\nVISUAL_RETRIEVAL_THRESHOLD_RELAXED=NO\nVISUAL_RETRIEVAL_PRODUCTION_COLLECTIONS_MUTATED=NO\nVISUAL_RETRIEVAL_TEMP_QDRANT=DELETED\n\nWORKER_LIFECYCLE_GPU_RECLAIM=PASS\nQDRANT_STORAGE_SEAL=PASS\nQDRANT_SNAPSHOT_PERSISTENT_COPY=NO\n\nFINAL EXECUTION ACCEPTANCE VERDICT : PASS\nPUBLIC_DEMO_EXTENDED_SCORECARD=PASS\n\nSEMANTIC_CORPUS_RETRIEVAL_PATHS_TOP1=36/36\nSEMANTIC_CORPUS_SEARCH_SPACE_ENTITIES=99967\n\nVISUAL_ROBUSTNESS_RETRIEVAL_PATHS_TOP1=32/32\nVISUAL_ROBUSTNESS_SEARCH_SPACE_IMAGES=4\n\nPUBLIC_DEMO_TOTAL_EXECUTED_RETRIEVAL_CHECKS=68/68\n```\n\n### Diễn giải score / Score interpretation\n\n- `image→text`: semantic cross-modal retrieval trên corpus production 99,967 entities; raw cosine thấp hơn là bình thường trong không gian cross-modal.\n- `image→image`: visual identity / robustness retrieval trên temporary 4-image gallery; transformed query và original target cùng modality nên `0.90+` là mục tiêu hợp lý.\n- `68/68` là tổng số **retrieval checks đã thực thi**, không phải một benchmark duy nhất trên corpus 99,967.\n- Không phép rescale nào được dùng để biến cosine thành số đẹp hơn.\n"))
